In [0]:
%run ../00_utils

In [0]:
def load_to_bronze(date, batch_id, load_type):
  print("Loading", date, "with batch_id", batch_id, "and load_type", load_type)
  year = date[:4]
  src = "/Volumes/workspace/bronze/raw_landing/mkt_summary/" + year + "/" + date + ".Z"
  out = "/Volumes/workspace/bronze/raw_landing/_extracted/" + year + "/" + date + ".txt"

  # 1. Decompress .Z (ZIP or GZIP)
  data = open(src, "rb").read()
  if data[:2] == b"PK":
      z = zipfile.ZipFile(io.BytesIO(data))
      text = z.read(z.namelist()[0]).decode("latin-1")
  else:
      text = gzip.decompress(data).decode("latin-1")

  # 2. Split lines: correct number of fields -> good, anything else -> bad   # NEW
  good_lines, bad_lines = [], []
  for line in text.splitlines():
      if line.strip() == "":
          continue                                  # ignore blank lines
      n = line.count("|")
      if n == EXPECTED_PIPES:
          good_lines.append(line)
      else:
          bad_lines.append((line, f"expected 13 fields, got {n + 1}"))

  # 3. Write only good lines to staging
  os.makedirs(os.path.dirname(out), exist_ok=True)
  with open(out, "w", encoding="utf-8") as f:
      f.write("\n".join(good_lines))

  print("Good lines:", len(good_lines), "| bad lines:", len(bad_lines))

  # 4. Quarantine: clear this date's old entries, then write the new ones   # NEW
  spark.sql(f"""
      DELETE FROM workspace.ops.quarantine_mkt_summary
      WHERE file_trade_date = '{date}' AND layer = 'raw_to_bronze'
  """)
  if bad_lines:
      now = dt.datetime.now()
      day = dt.date.fromisoformat(date)
      rows = [(line, reason, "raw_to_bronze", src, day, batch_id, now)
              for line, reason in bad_lines]
      (spark.createDataFrame(rows, quarantine_schema)
            .write.mode("append")
            .saveAsTable("workspace.ops.quarantine_mkt_summary"))

  # 5. Read good lines with the explicit schema
  df = (spark.read
        .schema(mkt_summary_schema)
        .option("sep", "|")
        .option("header", "false")
        .csv(out))

  # 6. Add label columns
  df_bronze = (df
    .withColumn("file_trade_date", F.lit(date).cast("date"))
    .withColumn("source_file",     F.lit(src))
    .withColumn("batch_id",        F.lit(batch_id))
    .withColumn("load_timestamp",  F.current_timestamp())
  ).select(
    "trade_date", "symbol", "sector_code", "company_name",
    "open_raw", "high_raw", "low_raw", "close_raw", "volume_raw", "ldcp_raw",
    "extra_1", "extra_2", "extra_3",
    "file_trade_date", "source_file", "batch_id", "load_timestamp"
  )

  # 7. Write to Bronze (replace this date only)
  (df_bronze.write
     .mode("overwrite")
     .option("replaceWhere", f"file_trade_date = '{date}'")
     .saveAsTable("workspace.bronze.mkt_summary"))

  rows_loaded = df_bronze.count()
  print("Loaded", rows_loaded, "rows for", date)
  return rows_loaded, len(bad_lines)  

In [0]:
def load_range(start_date, end_date, load_type):

    batch_id = f"bronze_{dt.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
    day  = dt.date.fromisoformat(start_date)
    last = dt.date.fromisoformat(end_date)

    total_rows = 0
    total_quarantined = 0
    days_loaded = 0
    days_missing = 0
    days_failed = 0

    while day <= last:
        d = day.isoformat()                 # e.g. "2015-01-02"

        if day.weekday() < 5:               # Mon-Fri only (5 = Sat, 6 = Sun)
            year = d[:4]
            src = "/Volumes/workspace/bronze/raw_landing/mkt_summary/" + year + "/" + d + ".Z"

            if os.path.exists(src):
                start = dt.datetime.now()
                try:
                    rows, quarantined = load_to_bronze(d, batch_id, load_type)
                    write_log(batch_id, "raw_to_bronze", load_type, src,
                              start, dt.datetime.now(), "SUCCESS",
                              rows_inserted=rows, rows_quarantined=quarantined)
                    total_rows += rows
                    total_quarantined += quarantined
                    days_loaded += 1
                except Exception as e:
                    write_log(batch_id, "raw_to_bronze", load_type, src,
                              start, dt.datetime.now(), "FAILURE",
                              error_message=str(e)[:1000])
                    print("FAILED:", d, "-", str(e)[:200])
                    days_failed += 1
            else:
                print("No file (holiday or not uploaded):", d)
                days_missing += 1

        day = day + dt.timedelta(days=1)

    print(f"Done. batch={batch_id}, days loaded={days_loaded}, "
          f"failed={days_failed}, weekdays without file={days_missing}, "
          f"total rows={total_rows}, quarantined={total_quarantined}")
    return batch_id